In [17]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import f1_score, make_scorer, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Joining datasets on `patient_id`

In [18]:
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")
complaints_df = pd.read_csv("dataset/chief_complaints.csv")
histsory_df = pd.read_csv("dataset/patient_history.csv")

patient_info_df = complaints_df.merge(
    histsory_df,
    on="patient_id",
    how="outer"
)

train_full_df = train_df.merge(
    patient_info_df,
    on="patient_id",
    how="left"
)

test_full_df = test_df.merge(
    patient_info_df,
    on="patient_id",
    how="left"
)

train_full_df.to_csv("dataset/train_dataset.csv", index=False)
test_full_df.to_csv("dataset/test_dataset.csv", index=False)

train_df = train_full_df
test_df = test_full_df

# Train & Test datasets overview

In [19]:
print(f"train_df shape:\n{train_df.shape}\n")
print(f"test_df shape:\n{test_df.shape}\n")


# print(f"train_df dtypes:\n{train_df.dtypes.to_string()}")
# print(f"test_df dtypes:\n{test_df.dtypes.to_string()}")

train_fields = train_df.columns
test_fields = test_df.columns

missing_fields = [field for field in train_fields if field not in test_fields]
print(f"test_df is missing {missing_fields} attributes")

print(f"{train_df.loc[0].to_string()}")

train_df shape:
(80000, 67)

test_df shape:
(20000, 64)

test_df is missing ['disposition', 'ed_los_hours', 'triage_acuity'] attributes
patient_id                                                         TG-UXRGA9UCO
site_id                                                             SITE-TMP-01
triage_nurse_id                                                      NURSE-0033
arrival_mode                                                            walk-in
arrival_hour                                                                  6
arrival_day                                                              Monday
arrival_month                                                                 5
arrival_season                                                           spring
shift                                                                   morning
age                                                                          43
age_group                                                       

# DBG files containing attributes informations

In [20]:
of = open("DBG/categorical_describe.txt", "w")
of.write(train_df.describe(include=[object]).to_string())
of.close()

of = open("DBG/numerical_describe.txt", "w")
of.write(train_df.describe(exclude=[object]).to_string())
of.close()

of = open("DBG/nans_per_col.txt", "w")
of.write(train_df.isna().sum(axis=0).to_string())
of.close()

nans_per_row = train_df.isna().sum(axis=1)
mean = nans_per_row.mean()
std = nans_per_row.std()

print(f"the average number of missing values per row is: {mean}, with a standard deviation of {std}")

the average number of missing values per row is: 0.3046375, with a standard deviation of 1.1407233491148236


The columns containing the highest number of missing values are:
- systolic_bp
- diastolic_bp
- mean_arterial_pressure
- pulse_pressure
- shock_index

all of which have 4146 nans (~ 5% of all samples)

---

Some discussion can be done on categorical features, as some of them have few unique values (even though many are justified, like `sex` and `season`) and others have many unique values but one of these represents half of the samples (like `insurance_type` which is equal to `public` in ~60% of the population)

---

As for numerical attributes, some have very low variance (< 1), but that could depend on the unit of measure and the intrinsic meaning of the attribute, thus further discussion must be performed.

# Mock pipeline

In [ ]:
y_train = train_df['triage_acuity']
train_df_clean = train_df.drop(columns=['patient_id','disposition', 'ed_los_hours', 'triage_acuity', 'chief_complaint_raw'])


categoricals = [field for field in train_df_clean.columns if train_df_clean.dtypes[field] == object]
numericals = [field for field in train_df_clean.columns if train_df_clean.dtypes[field] != object]

preprocessor = ColumnTransformer(
    transformers = [
        (
            "cat",
            Pipeline(
                [
                    ("imp", SimpleImputer(strategy='most_frequent')),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
                ]
            ),
            categoricals
        ),
        (
            "num",
            Pipeline(
                [
                    ("imp", SimpleImputer()),
                    ("scal", StandardScaler())
                ]
            ),
            numericals
        )
    ],
    remainder = "drop"
)

X_train, X_val, y_train_split, y_val = train_test_split(train_df_clean, y_train, train_size=0.6, random_state=42, stratify=y_train)

print(f"starting preprocessing...")

X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp = preprocessor.transform(X_val)

print(f"final dimensionality: {X_train_pp.shape}")

clf = RandomForestClassifier(
    n_estimators = 100,
    criterion = 'log_loss',
    max_depth = 25,
    min_samples_split = 10,
    min_samples_leaf = 2,
    max_features = 'sqrt',
    n_jobs = -1,
    random_state = 42
)

print(f"starting fitting...")
clf.fit(X_train_pp, y_train_split)

print(f"fitting done, starting predicting...")
y_pred = clf.predict(X_val_pp)

acc = accuracy_score(y_val, y_pred)
print(f"predicting done, overall accuracy: {acc}")

0        2
1        5
2        5
3        3
4        3
        ..
79995    4
79996    3
79997    5
79998    1
79999    4
Name: triage_acuity, Length: 80000, dtype: int64
categorical features:
 ['site_id', 'triage_nurse_id', 'arrival_mode', 'arrival_day', 'arrival_season', 'shift', 'age_group', 'sex', 'language', 'insurance_type', 'transport_origin', 'pain_location', 'mental_status_triage', 'chief_complaint_system_x', 'chief_complaint_system_y']
numerical features:
['arrival_hour', 'arrival_month', 'age', 'num_prior_ed_visits_12m', 'num_prior_admissions_12m', 'num_active_medications', 'num_comorbidities', 'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score', 'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score', 'hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx

fitting done, starting predicting...
predicting done, overall accuracy: 0.82115625
